# nano Beatrix — the frame grid on a rented card

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_frame_grid.ipynb)

**Crafts**: the nano twin pair at their bank of record, the byte-class table beside the aleph address. **Castor** is the rotary softmax body (`castor-both`, 9.2M at 4,096), **Pollux** the alternating body whose hub half keeps a constant decode memory (`pollux-both`, 8.9M). Each craft trains from birth for 12,000 steps on dynamically framed rows: frames of generic sizes opened and closed by the byte registry's tags, some nested, with a question that can land on any frame. The record of every run carries the held-out bits inside each frame by kind, size and segment, the fetch by frame size and depth, the plain-text bits beside, and the cost.

**The cells, in order**: C1 pins the trainer and holds the decision block; C2 builds the data, prints the plan, runs 200 steps on this card for the price line and probes the push path; C3 is the session, resume-first, with a grid bar and a step bar, every finished run shipped to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) as it lands; C4 renders the results table.

**Before C1**: put a Hugging Face token with write access to that repo in this notebook's secrets as `HF_TOKEN` and allow the notebook to read it. Tokenless, everything runs and nothing ships. A file named `STOP` in the session folder ends the grid between runs; a restarted session resumes where it stopped. The trainer and its tests live in [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.13"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 13), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed: 160 MB covers 12,000 steps x 8 x 4,096 four times over
GRID = dict(
    arms='castor-both,pollux-both',   # the twins at the decided bank (table + address, 2026-10-12): the best text on both bodies on two seeds
    seeds='1,2',                      # the two-seed rule
    register='caption,math,code',     # C1 = the register grid: letters, digits and hex values under their own register tags
    ctx=4096, mb=8, steps=12000,      # the recipe the bank decision was read at: 12,000 steps x 8 rows x 4,096
    frames='dyn', plant=0.75,         # three rows in four framed (dynamic sectors of generic sizes with random gaps), one in four a plain document:
                                      # an all-framed diet made a frame specialist of one Castor seed that could not read a bare document (2026-10-13 probe)
    sizes='32,128,256,512',           # the generic sizes; 512 enters at 4,096
    frame_records=4, nest=0.3, overlap=0.0, ask='any', tags='mode',   # four records a lookup frame, nesting 0.3, no overlap in the first pass, the question on any frame, MODE + letter tags
    log_every=100,                    # the trainer's progress line (step, bits, tok/s, time left) feeds the step bar
)
SMOKE_STEPS = 200                     # C2's short run on this card for the price line; its push is off
PARALLEL = 4                          # runs at once on the card: a 9M craft at 8 x 4,096 takes ~5 GB and a fraction of a 96 GB card's compute (RTX PRO 6000,
                                      # 850k tok/s alone, 2026-10-13); the step bars' per-run tok/s is the measurement: drop to 2 if four runs sum under one run's rate
print('decision block:', len(GRID), 'keys | out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, a short run on this card (the price line), the push probe. No grid training.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plan: what the grid expands to, and what this folder already holds
plan = bed_runner.run(GRID, out=OUT, dry=True)

# the price line: the first arm of the grid for SMOKE_STEPS on this card (the step bar runs; the record stays local under its own short-run name)
smoke = dict(GRID, arms=GRID['arms'].split(',')[0], seeds='1', register=GRID['register'].split(',')[0], steps=SMOKE_STEPS, log_every=20)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any grid'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = GRID['steps'] * GRID['ctx'] * GRID['mb'] / (last['tok_s'] * 1e3) / 60
todo = plan['runs'] - plan['done_before']
print(f"this card: {last['tok_s']}k tok/s at {GRID['ctx']} x {GRID['mb']} -> one {GRID['steps']:,}-step run about {per_run:.0f} min; the {todo} runs left about {per_run * todo / 60:.1f} h")

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'grid': GRID}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano frame grid: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE SESSION: the grid, PARALLEL runs at once, resume-first (runner_done.txt in OUT); a STOP file in OUT starts no new run and
# lets the running ones finish; a grid bar over the runs and a step bar per running child fed by the trainer's own progress lines; the job-size and gauge lines print through the bars; every finished run
# ships to REPO as it lands. An interrupt loses only the running run (the bed has no mid-run checkpoint); rerun the cell to continue.
from geolip.nano_beatrix import bed_runner
res = bed_runner.run(GRID, out=OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', parallel=PARALLEL)
print('session:', {k: v for k, v in res.items() if k != 'ran'}, '| ran', len(res['ran']))

In [ ]:
# C4 — THE RESULTS: the index over this session's records; the framed rows table carries the text, the bits outside the frames, the data
# frames' body bits by size, the whole value fetched from a frame by its size, the nested whole and the cost. Run any time; the Hub holds the same records.
from IPython.display import Markdown, display
from geolip.nano_beatrix.bed_table import render
display(Markdown(render(f'{OUT}/bed', pairs=True)))

### The other grids

Change `GRID` in C1, rerun C1 (the install is cached), then C3; finished runs are skipped.

| grid | the change in `GRID` |
|---|---|
| C2 the in-window capacity axis | `seeds='1'`, `register='caption'`, `frame_records='4,8,16'`, `sizes='32,128,256,512,1024'` |
| C3 the weights source | `seeds='1'`, `register='caption'`, `facts='64,256,1024'`, `ask='weights'` |
| C4 the scrutiny set | `overlap=0.3`, `nest=0.6`, `ask='nested,overlap'` |
| the all-framed diet (task crafts that always see frames) | `plant=1.0`; read the plain-document bits beside the frame gauges, they are the canary |

A smaller card: `mb=4` at 4,096 (the step count stays; a run takes longer). The session folder is the Colab disk: the Hub copy is the record; a new session rebuilds the corpus in C2 and resumes nothing unless `/content/out/runner_done.txt` is restored.